# Introduction to Machine Learning — Lecture 1

Unit 2 · ML Fundamentals & Preprocessing

Files needed: `Titanic-Dataset.csv`, `tips.csv`

In [ ]:
! pip3 install scikit-learn


[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import numpy as np
import pandas as pd
import sklearn

---
## 1. What is Machine Learning

> **Flow:** Rules written by hand → rules learned from data.

### Practice 1

**Q1.** Gmail decides which mails go to the Spam folder.

- Version A: a rule is written — any mail containing the word "lottery" goes to Spam.
- Version B: millions of users have clicked "Report spam" over the years, and Gmail works
  out for itself what spam looks like.

Which is traditional programming, which is machine learning?

<details><summary>Answer</summary>

A is traditional programming — a person wrote the condition.
B is machine learning — the mails and the past answers are given, and the rule is learned.
</details>

**Q2.** Your phone unlocks when it sees your face. Why would it be very hard to write that
as a set of conditions by hand?

<details><summary>Answer</summary>

A face looks different with light, angle, glasses, a haircut, a beard. There is no small set
of conditions covering all of it — too many cases to write by hand, which is when ML is useful.
</details>

---
## 2. Supervised vs Unsupervised

> **Flow:** Does the data have an answer column, or not?

Today is supervised.

### Practice 2

**Q1.** Swiggy has records of every order it has delivered, and each record says whether that
delivery was late or on time. They want to predict this for a new order.
Supervised or unsupervised?

<details><summary>Answer</summary>

Supervised — the answer column already exists in the records.
</details>

**Q2.** Spotify has the listening history of every user. Nobody has ever put users into
categories. The company wants to find out whether natural groups of listeners exist.
Supervised or unsupervised?

<details><summary>Answer</summary>

Unsupervised — there is no answer column, so the goal is to find groups, not predict a
known answer.
</details>

---
## 3. Classification vs Regression

> **Flow:** Look at the target column, and only the target column.

Number in a range → regression. One of a fixed set → classification.

In [6]:
df = pd.read_csv('/Users/nikunj/machine learning/Introduction_To_Machine_Learning/Data/Titanic-Dataset.csv')
df['Survived'].value_counts()

Survived
0    549
1    342
Name: count, dtype: int64

### Practice 3

Classification or regression?

**Q1.** Ola shows you an estimated fare of Rs 247 before you book the ride.

**Q2.** GPay flags a transaction as suspicious or not suspicious.

**Q3.** Gmail sorts an incoming mail into one of three tabs — Primary, Social, or Promotions.

<details><summary>Answers</summary>

1. Regression — the fare is a number in a range.
2. Classification — two labels.
3. Classification — three labels. Numbering the tabs 1, 2, 3 would not make it regression.
</details>

---
## 4. The ML Pipeline

> **Flow:** Six stages. Every lecture in this course sits inside one of them.

`1 Clean → 2 Split → 3 Feature Engineering → 4 Train → 5 Predict → 6 Evaluate`

Stages 1–3 are most of the work.

---
## 5. scikit-learn

> **Flow:** One tool per stage. Two patterns.

Preprocessor: `.fit()` → `.transform()`
Model: `.fit()` → `.predict()`


## 6. Stage 1 — Data Cleaning

> **Flow:** Load, find what is broken, fix it.

**Question:** given a passenger's details, did they survive?

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [7]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Out of 891 rows: `Age` has 714, `Cabin` has 204, `Embarked` has 889.

In [8]:
# Drop unusable columns, fill Age, drop 2 rows


`Cabin` is 77% empty. `Name`, `Ticket`, `PassengerId` are unique per row, so no pattern.

In [9]:

df=df.drop(columns=["Cabin","Name","Ticket","PassengerId"])


In [10]:
df["Age"]=df["Age"].fillna(df["Age"].median())
df.dropna(subset=["Embarked"],inplace=True)

In [11]:
df.info()

<class 'pandas.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  889 non-null    int64  
 1   Pclass    889 non-null    int64  
 2   Sex       889 non-null    str    
 3   Age       889 non-null    float64
 4   SibSp     889 non-null    int64  
 5   Parch     889 non-null    int64  
 6   Fare      889 non-null    float64
 7   Embarked  889 non-null    str    
dtypes: float64(2), int64(4), str(2)
memory usage: 62.5 KB


In [12]:
# Features and target

x=df.drop(columns=["Survived"]) # independent features
y=df["Survived"] # dependent Features 
print(x.head(1))
print(y.head(1))

   Pclass   Sex   Age  SibSp  Parch  Fare Embarked
0       3  male  22.0      1      0  7.25        S
0    0
Name: Survived, dtype: int64


---
## 8. Stage 2 — Train/Test Split

> **Flow:** Hide some rows before preparing anything.

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
#  Split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)


In [15]:
print(x_test.shape)
print(x_train.shape)


(178, 7)
(711, 7)


`random_state` — same rows for everyone. `stratify` — same survival ratio in both halves.

In [16]:
from sklearn.neighbors import KNeighborsClassifier
model=KNeighborsClassifier()
model.fit(x_train,y_train)


ValueError: could not convert string to float: 'female'

In [18]:
x_train.head(1)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
708,1,female,22.0,0,0,151.55,S


---
## 9. Stage 3 — Encoding

> **Flow:** Turn text columns into numbers.

Nominal → one-hot. Ordinal → numbered in order. Label → target only.

In [19]:
# Fit the encoder on training rows only
from sklearn.preprocessing import OneHotEncoder

---
## 10. Stage 3 — Encoding in Code

> **Flow:** One object converts the text columns and leaves the rest alone.

`ColumnTransformer` — pick columns, pick a tool for them.

In [20]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer


In [21]:
cat_cols = ["Sex", "Embarked"]


In [22]:
ct=ColumnTransformer([('cat',OneHotEncoder(sparse_output=False),cat_cols)],
                     remainder='passthrough')
x_train_final=ct.fit_transform(x_train)
x_test_final=ct.transform(x_test)


In [23]:
x_train.shape
x_train_final.shape


(711, 10)

7 columns → 10. First five values are the one-hot flags, rest are the original numbers.

- `fit_transform` on train, `transform` on test — never `fit` on test.
- `remainder='passthrough'` keeps unlisted columns. Default is `'drop'`, which deletes them silently.
- Output is a NumPy array, so column names are gone. `ct.get_feature_names_out()` brings them back.

### Practice 4

**Q1.** A restaurant stores `sex` (Male/Female), `smoker` (Yes/No), `day` (Thu/Fri/Sat/Sun)
and `time` (Lunch/Dinner). Which of these has a real order? How many columns does the table
gain if all four are one-hot encoded?

<details><summary>Answer</summary>

None of them has a real order — Thursday is not "less" than Sunday for this purpose.
One-hot gives 2 + 2 + 4 + 2 = 10 new columns, replacing the original 4.
</details>

**Q2.** A shop's `size` column holds Small, Medium, Large. Should this be one-hot encoded?

<details><summary>Answer</summary>

No — there is a real order here, so ordinal encoding (1, 2, 3) is correct and keeps that order.
One-hot would throw the order away.
</details>

---
## 11. Stage 4 & 5 — Train and Predict

> **Flow:** `.fit()` learns, `.predict()` answers.

KNN: measure distance to every training row, take the `k` closest, majority wins.

In [24]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [25]:
model=KNeighborsClassifier()
model.fit(x_train_final,y_train)
y_pre=model.predict(x_test_final)
print(accuracy_score(y_test,y_pre))

0.6853932584269663


In [26]:
from sklearn.preprocessing import StandardScaler

In [27]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train_final)
x_test_scaled = scaler.transform(x_test_final)


In [28]:
scaler_model = KNeighborsClassifier()
scaler_model.fit(x_train_scaled, y_train)
y_pre_scaled = scaler_model.predict(x_test_scaled)
print("Accuracy (without scaling):", accuracy_score(y_test, y_pre))
print("Accuracy (with scaling):", accuracy_score(y_test, y_pre_scaled))

Accuracy (without scaling): 0.6853932584269663
Accuracy (with scaling): 0.7921348314606742


In [31]:
for k in [3,5,7,11,13,15,21,25,29]:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(x_train_scaled,y_train)
    m_pre=m.predict(x_test_scaled)
    print(k,round(accuracy_score(y_test,m_pre),2))

3 0.78
5 0.79
7 0.78
11 0.78
13 0.79
15 0.81
21 0.83
25 0.81
29 0.81
